# Computación Adiabática y QAOA Avanzado

**Laboratorio 35 · Nivel muy avanzado**

La computación adiabática explota el **teorema adiabático**: si un sistema cuántico
evoluciona suficientemente despacio desde el estado fundamental de $H_0$ hasta el de
$H_P$, permanece en el estado fundamental en todo momento.

$$H(s) = (1-s)H_0 + s\,H_P \quad s \in [0,1]$$

Este laboratorio conecta la evolución adiabática con QAOA (su versión Trotterizada)
y estudia cuándo uno supera al otro.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from scipy.optimize import minimize
import warnings; warnings.filterwarnings('ignore')

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator

print('Librerías cargadas OK')


## 1. Teorema adiabático: tiempo mínimo

El teorema adiabático garantiza $\|\langle n(t)|\psi(t)\rangle - 1\| < \varepsilon$ si:

$$T \gg \frac{\max_{s} |\langle n(s)|\dot{H}(s)|m(s)\rangle|}{\Delta(s)^2} \quad \Delta(s) = E_1(s) - E_0(s)$$


In [ ]:
def gap_adiabatico(H0: np.ndarray, HP: np.ndarray, n_pts: int = 200) -> tuple:
    """Calcula el gap energético y los autovalores a lo largo del camino adiabático."""
    s_vals = np.linspace(0, 1, n_pts)
    gaps = []; E0s = []; E1s = []
    for s in s_vals:
        H = (1 - s) * H0 + s * HP
        eigvals = np.linalg.eigvalsh(H)
        E0s.append(eigvals[0]); E1s.append(eigvals[1])
        gaps.append(eigvals[1] - eigvals[0])
    return np.array(s_vals), np.array(gaps), np.array(E0s), np.array(E1s)

# MAX-CUT en triángulo: H_P = -0.5*(ZZ_01 + ZZ_12 + ZZ_02) + 1.5*III
# (buscamos el corte de mayor energía = mínimo de -MAX-CUT)
H_mixer = SparsePauliOp.from_list([('XII', -1.0), ('IXI', -1.0), ('IIX', -1.0)])
H_maxcut = SparsePauliOp.from_list([('ZZI', -0.5), ('IZZ', -0.5), ('ZIZ', -0.5), ('III', 1.5)])
H0 = H_mixer.to_matrix().real; HP = H_maxcut.to_matrix().real

s_vals, gaps, E0s, E1s = gap_adiabatico(H0, HP)
delta_min = gaps.min()
s_min = s_vals[np.argmin(gaps)]

print(f'Gap mínimo: Δ_min = {delta_min:.4f} en s = {s_min:.3f}')
print(f'Tiempo adiabático mínimo estimado: T_min ~ 1/Δ² = {1/delta_min**2:.1f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(s_vals, E0s, 'b-', lw=2, label='E₀ (estado fundamental)')
ax.plot(s_vals, E1s, 'r--', lw=2, label='E₁ (primer excitado)')
ax.fill_between(s_vals, E0s, E1s, alpha=0.15, color='g', label='Gap Δ(s)')
ax.axvline(s_min, color='orange', ls=':', lw=2, label=f'Δ_min @ s={s_min:.2f}')
ax.set_xlabel('Parámetro adiabático s'); ax.set_ylabel('Energía')
ax.set_title('Espectro del camino adiabático: Mixer → MAX-CUT (triángulo 3q)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 2. Simulación adiabática via Trotter

Dividimos el tiempo total $T$ en $r$ pasos de longitud $\delta t = T/r$:

$$U(\delta t, s) \approx e^{-i H(s) \delta t} \approx e^{-i H_0 (1-s)\delta t} e^{-i H_P s\,\delta t}$$


In [ ]:
def evolucion_adiabatica(H0: np.ndarray, HP: np.ndarray, T: float, r: int) -> np.ndarray:
    """
    Simula la evolución adiabática de H0 a HP en tiempo T con r pasos Trotter.
    Retorna el estado final.
    """
    N = H0.shape[0]
    psi = np.zeros(N, dtype=complex)
    psi[0] = 1.0  # estado fundamental de H0 = |+>^3 (H mixer)
    # Preparar estado inicial correcto: estado fundamental de H0
    E0_init, V0 = np.linalg.eigh(H0)
    psi = V0[:, 0].copy()

    dt = T / r
    for k in range(r):
        s = (k + 0.5) / r  # punto medio del intervalo
        U0 = expm(-1j * H0 * (1-s) * dt)
        UP = expm(-1j * HP * s * dt)
        psi = UP @ U0 @ psi
    return psi

# Energía del estado fundamental de HP
E_ground, V_ground = np.linalg.eigh(HP)
E_target = E_ground[0]

# Barrido en T
T_vals = np.logspace(0, 3, 40)
fidelidades_T = []

for T in T_vals:
    r = max(10, int(10 * T))  # r proporcional a T
    psi_final = evolucion_adiabatica(H0, HP, T, r)
    fid = abs(psi_final.conj() @ V_ground[:, 0])**2
    fidelidades_T.append(fid)

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(T_vals, fidelidades_T, 'b-o', ms=4, lw=2)
ax.axhline(0.99, color='g', ls='--', lw=1.5, label='Fidelidad = 0.99')
ax.axvline(1/delta_min**2, color='r', ls=':', lw=2, label=f'T ~ 1/Δ² = {1/delta_min**2:.0f}')
ax.set_xlabel('Tiempo total T'); ax.set_ylabel('Fidelidad con estado fundamental')
ax.set_title('Evolución adiabática: MAX-CUT triángulo 3q')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

T99_idx = next((i for i, f in enumerate(fidelidades_T) if f > 0.99), -1)
print(f'Fidelidad > 0.99 conseguida para T ≥ {T_vals[T99_idx]:.1f}')


## 3. QAOA como versión discreta del annealing

QAOA con $p$ capas implementa $p$ pasos de Trotter con ángulos optimizables:

$$|\psi(\beta,\gamma)\rangle = \prod_{k=1}^p e^{-i\beta_k H_0} e^{-i\gamma_k H_P} |+\rangle^{\otimes n}$$


In [ ]:
def qaoa_circuit(n: int, gamma_vals: list, beta_vals: list,
                 edges: list) -> QuantumCircuit:
    """QAOA para MAX-CUT con p capas."""
    p = len(gamma_vals)
    qc = QuantumCircuit(n)
    qc.h(range(n))  # estado inicial |+>^n

    for layer in range(p):
        # Fase de problema
        gamma = gamma_vals[layer]
        for i, j in edges:
            qc.cx(i, j); qc.rz(2*gamma, j); qc.cx(i, j)
        # Fase de mixer
        beta = beta_vals[layer]
        qc.rx(2*beta, range(n))

    return qc

edges = [(0,1), (1,2), (0,2)]  # triángulo
estimator = StatevectorEstimator()

def qaoa_energia(params: np.ndarray, p: int) -> float:
    gamma_vals = params[:p]; beta_vals = params[p:]
    qc = qaoa_circuit(3, gamma_vals, beta_vals, edges)
    return float(estimator.run([(qc, H_maxcut, [])]).result()[0].data.evs)

# Optimizar QAOA para p = 1, 2, 3, 4
print('Optimizando QAOA para MAX-CUT en triángulo...')
resultados_qaoa = {}
E_clasico_max = -E_target  # MAX-CUT óptimo = -E_min(-MAX-CUT)

for p in [1, 2, 3, 4]:
    mejor = None
    for seed in range(8):
        rng = np.random.default_rng(seed)
        x0 = rng.uniform(0, np.pi, 2*p)
        res = minimize(qaoa_energia, x0, args=(p,), method='COBYLA',
                       options={'maxiter': 500})
        if mejor is None or res.fun < mejor.fun:
            mejor = res
    E_opt = mejor.fun
    ratio = (-E_opt) / E_clasico_max
    resultados_qaoa[p] = {'E': E_opt, 'ratio': ratio}
    print(f'  p={p}: E={E_opt:.4f} | Razón aprox. = {ratio:.4f} (óptimo = 1.0)')


In [ ]:
# Comparativa QAOA vs adiabático
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Razón de aproximación
p_vals = list(resultados_qaoa.keys())
ratios = [resultados_qaoa[p]['ratio'] for p in p_vals]
axes[0].plot(p_vals, ratios, 'b-o', ms=8, lw=2)
axes[0].axhline(1.0, color='g', ls='--', lw=1.5, label='Óptimo')
axes[0].axhline(0.75, color='r', ls=':', lw=1.5, label='Cota clásica 0.75')
axes[0].set_xlabel('Capas QAOA p'); axes[0].set_ylabel('Razón de aproximación')
axes[0].set_title('QAOA MAX-CUT: razón de aproximación vs p')
axes[0].set_ylim(0.7, 1.05); axes[0].legend(); axes[0].grid(alpha=0.3)

# QAOA vs adiabático — fidelidad
E_qaoa_p_vals = [resultados_qaoa[p]['E'] for p in p_vals]
fid_adiab_equiv = [abs(evolucion_adiabatica(H0, HP, p*5, p*50).conj() @ V_ground[:,0])**2
                   for p in p_vals]

axes[1].plot(p_vals, [-e/E_clasico_max for e in E_qaoa_p_vals], 'b-o', ms=8, lw=2, label='QAOA')
axes[1].plot(p_vals, fid_adiab_equiv, 'r-s', ms=8, lw=2, label='Adiabático (T=5p)')
axes[1].set_xlabel('Capas / pasos p'); axes[1].set_ylabel('Calidad (↑ mejor)')
axes[1].set_title('QAOA vs Adiabático: misma profundidad de circuito')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 4. Programación adiabática de problemas de optimización

MAX-CUT, 3-SAT y partición de números se mapean a Hamiltonianos Ising.
El annealing cuántico (D-Wave) minimiza el Hamiltoniano Ising:

$$H_{\text{Ising}} = \sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j$$


In [ ]:
def maxcut_to_ising(n: int, edges: list, pesos: list = None):
    """Convierte MAX-CUT a Hamiltoniano Ising. Retorna SparsePauliOp."""
    if pesos is None: pesos = [1.0] * len(edges)
    terms = []
    offset = sum(pesos) / 2
    for (i, j), w in zip(edges, pesos):
        op = 'I'*min(i,j) + 'Z' + 'I'*(abs(i-j)-1) + 'Z' + 'I'*(n-max(i,j)-1)
        terms.append((op, -w/2))
    terms.append(('I'*n, offset))
    return SparsePauliOp.from_list(terms)

# Grafo completo K4 (6 aristas)
n4 = 4
edges4 = [(i,j) for i in range(n4) for j in range(i+1,n4)]
H_k4 = maxcut_to_ising(n4, edges4)

E_k4 = np.linalg.eigvalsh(H_k4.to_matrix().real)[0]
print(f'MAX-CUT K4: E_min = {E_k4:.4f} (MAX-CUT óptimo = {-E_k4 + 3:.1f} aristas)')

# Verificar con QAOA p=3
def qaoa_k4(params):
    p = len(params) // 2
    qc = qaoa_circuit(n4, params[:p], params[p:], edges4)
    return float(estimator.run([(qc, H_k4, [])]).result()[0].data.evs)

best_k4 = None
for seed in range(10):
    rng = np.random.default_rng(seed)
    res = minimize(qaoa_k4, rng.uniform(0,np.pi,6), method='COBYLA',
                   options={'maxiter':400})
    if best_k4 is None or res.fun < best_k4.fun: best_k4 = res

cut_qaoa = -best_k4.fun + 3
print(f'QAOA p=3 K4: corte = {cut_qaoa:.3f} aristas (máximo = 4)')


## Conclusiones

- El **teorema adiabático** garantiza el estado fundamental si $T \gg 1/\Delta^2_{\min}$.
- **QAOA con $p$ capas** es la versión Trotterizada del annealing; para $p\to\infty$ converge al resultado adiabático.
- Para $p=1$, QAOA-MAX-CUT supera la cota clásica de 0.5 en grafos 3-regulares.
- La ventaja cuántica real de QAOA sobre algoritmos clásicos **sigue abierta** para $p$ finito.
